目标网址：https://movie.douban.com/top250

目标字段：
- 电影名称
- 导演
- 主演
- 电影放映年份、
- 电影拍摄国家
- 电影类型
- 评分
- 评价数 
- 电影名言
- 电影封面地址

# 定义url地址

In [ ]:
url='https://movie.douban.com/top250?start=0&filter='

# 发送请求

In [ ]:
import requests

In [ ]:
#添加请求头
headers={
    'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:106.0) Gecko/20100101 Firefox/106.0',
    'Cookie':__import__('os').environ.get('SCRAPER_COOKIE', '')
}

In [ ]:
reponse=requests.get(url,headers=headers)

In [ ]:
reponse.status_code

In [ ]:
reponse.text

# 解析源网页

## 电影名称

In [ ]:
from lxml import etree
html=etree.HTML(reponse.text)
movie_name1=html.xpath('//span[@class="title"]/text()')
movie_name=[]
for  i in movie_name1:
    if '\xa0/' in i:
        continue
    else:
        movie_name.append(i)
movie_name

## 导演

In [ ]:
temp=html.xpath('//p[@class=""]/text()')
temp

In [ ]:
temp_1=temp[::2]
temp_1

In [ ]:
movie_doctor=[]
for i in temp_1:
    temp_2=''
    temp_2+=i
    i=i.strip()
    class_1=i.split('\xa0\xa0\xa0')
    movie_doctor.append(class_1[0])
movie_doctor

## 主演

In [ ]:
movie_actor=[]
for i in temp_1:
    temp_2=''
    temp_2+=i
    i=i.strip()
    class_1=i.split('\xa0\xa0\xa0')
    movie_actor.append(class_1[1])
movie_actor

## 电影放映年份

In [ ]:
temp_3=temp[1::2]
temp_3

In [ ]:
year=[]
for i in temp_3:
    temp_4=''
    temp_4+=i
    i=i.strip()
    class_2=i.split('\xa0/\xa0')
    year.append(class_2[0])
year

## 电影拍摄国家

In [ ]:
nation=[]
for i in temp_3:
    temp_4=''
    temp_4+=i
    i=i.strip()
    class_2=i.split('\xa0/\xa0')
    nation.append(class_2[1])
nation

## 电影类型

In [ ]:
mold=[]
for i in temp_3:
    temp_4=''
    temp_4+=i
    i=i.strip()
    class_2=i.split('\xa0/\xa0')
    mold.append(class_2[2])
mold

## 评分

In [ ]:
grade=html.xpath('//span[@class="rating_num"]/text()')
grade

## 评价数

In [ ]:
comment_num=html.xpath('//div[@class="star"]/span[4]/text()')
for i in range(len(comment_num)):
    comment_num[i]=comment_num[i].strip('人评价')
comment_num

## 电影名言

In [ ]:
inq=html.xpath('//span[@class="inq"]/text()')
inq

## 电影封面地址

In [ ]:
img=html.xpath('//div[@class="pic"]/a/img/@src')
img

# 数据的储存

In [ ]:
len(movie_name),len(movie_doctor),len(movie_actor),len(year),len(nation),len(mold),len(grade),len(comment_num),len(inq),len(img)

In [ ]:
import pandas as pd

In [ ]:
dataframe=pd.DataFrame([movie_name,movie_doctor,movie_actor,year,nation,mold,grade,comment_num,inq,img],index=[ '电影名称','导演','主演','电影放映年份','电影拍摄国家','电影类型','评分','评价数','电影名言','电影封面地址']).T
dataframe

In [ ]:
dataframe.to_excel('豆瓣_top250.xlsx')

# 翻页抓取

In [ ]:
movies_name=[]
movies_doctor=[]
movies_actor=[]
years=[]
molds=[]
nations=[]
grades=[]
comments_num=[]
inqs=[]
imgs=[]
none=['主演']

for page in range(1,11):
    url=f'https://movie.douban.com/top250?start={25*(page-1)}&filter='
    print(url)
    reponse=requests.get(url,headers=headers)
    html=etree.HTML(reponse.text)
    movie_name1=html.xpath('//span[@class="title"]/text()')
    movie_name=[]
    for  i in movie_name1:
        if '\xa0/' in i:
            continue
        else:
            movie_name.append(i)
    temp=html.xpath('//p[@class=""]/text()')
    temp_1=temp[::2]
    movie_doctor=[]
    for i in temp_1:
        temp_2=''
        temp_2+=i
        i=i.strip()
        class_1=i.split('\xa0\xa0\xa0')
        movie_doctor.append(class_1[0])
        temp_3=temp[1::2]
    movie_actor=[]
    for i in temp_1:
        temp_2=''
        temp_2+=i
        i=i.strip()
        class_1=i.split('\xa0\xa0\xa0')
        if none not in class_1:
            class_1.extend(none)
        movie_actor.append(class_1[1])
    year=[]
    for i in temp_3:
        temp_4=''
        temp_4+=i
        i=i.strip()
        class_2=i.split('\xa0/\xa0')
        year.append(class_2[0])
    nation=[]
    for i in temp_3:
        temp_4=''
        temp_4+=i
        i=i.strip()
        class_2=i.split('\xa0/\xa0')
        nation.append(class_2[1])
    mold=[]
    for i in temp_3:
        temp_4=''
        temp_4+=i
        i=i.strip()
        class_2=i.split('\xa0/\xa0')
        mold.append(class_2[2])
    grade=html.xpath('//span[@class="rating_num"]/text()')
    comment_num=html.xpath('//div[@class="star"]/span[4]/text()')
    for i in range(len(comment_num)):
        comment_num[i]=comment_num[i].strip('人评价')
    inq=[i.xpath('.//span[@class="inq"]/text()' )[0] if  i.xpath('.//span[@class="inq"]/text()') else ''  for i in html.xpath('//div[@class="info"]')]
    img=html.xpath('//div[@class="pic"]/a/img/@src')
    
    movies_name.extend(movie_name)
    movies_doctor.extend(movie_doctor)
    movies_actor.extend(movie_actor)
    years.extend(year)
    molds.extend(mold)
    nations.extend(nation)
    grades.extend(grade)
    comments_num.extend(comment_num)
    inqs.extend(inq)
    imgs.extend(img)
    
    print(f'{page}/10~~~{len(movies_name),len(movies_doctor),len(movies_actor),len(years),len(nations),len(molds),len(grades),len(comments_num),len(inqs),len(imgs)}',)

In [ ]:
dataframe=pd.DataFrame([movies_name,movies_doctor,movies_actor,years,nations,molds,grades,comments_num,inqs,imgs],index=[ '电影名称','导演','主演','电影放映年份','电影拍摄国家','电影类型','评分','评价数','经典台词','电影封面地址']).T
dataframe

In [ ]:
dataframe.to_excel('豆瓣_top250.xlsx')

# 详细版

## 抓取链接

In [ ]:
links=[]
for page in range(1,11):
    url=f'https://movie.douban.com/top250?start={25*(page-1)}&filter='
    print(url)
    reponse=requests.get(url,headers=headers)
    html=etree.HTML(reponse.text)
    link=html.xpath('//div[@class="pic"]/a/@href')
    links+=link
len(links)
links

## 获取电影名称

In [ ]:
for i in links:
    